<a href="https://colab.research.google.com/github/suyogy1-bot/Machine_Learning/blob/main/Credit_Card_Customer_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Card Customer Segmentation
### Unsupervised Machine Learning: K-Means vs Hierarchical Clustering vs DBSCAN

---

## 1. Project Overview

**Business problem:** A bank has thousands of credit card customers, and they do not all behave the same way. Some spend a lot, some barely use the card, and some mostly withdraw cash. Sending the same offer to everyone is wasteful.

**Goal:** Group customers into segments based on their card usage behaviour, so the bank can target each group with the right offer.

**Why unsupervised learning?** We have no column that says "this customer is type A or type B". There is no target to predict. We let the algorithms find the groups on their own.

**What we will do in this project:**

| Step | What we do |
|------|-----------|
| 1 | Load and understand the data |
| 2 | Clean the data |
| 3 | Explore the data (EDA) |
| 4 | Prepare features (log transform and scaling) |
| 5 | Apply **K-Means** |
| 6 | Apply **Hierarchical Clustering** |
| 7 | Apply **DBSCAN** |
| 8 | Compare the three algorithms |
| 9 | Profile the segments and give business recommendations |

## 2. About the Dataset

**Source:** Credit Card Dataset for Clustering (Kaggle, by Arjun Bhasin)

The file contains about 9,000 active credit card holders observed over the last 6 months. Each row is one customer and each column describes their behaviour.

| Column | Meaning |
|--------|---------|
| CUST_ID | Customer ID |
| BALANCE | Amount left in the account |
| BALANCE_FREQUENCY | How often the balance is updated (0 to 1) |
| PURCHASES | Total amount spent on purchases |
| ONEOFF_PURCHASES | Largest single purchase amount |
| INSTALLMENTS_PURCHASES | Amount spent in instalments |
| CASH_ADVANCE | Cash withdrawn using the card |
| PURCHASES_FREQUENCY | How often purchases are made (0 to 1) |
| ONEOFF_PURCHASES_FREQUENCY | How often one-off purchases are made (0 to 1) |
| PURCHASES_INSTALLMENTS_FREQUENCY | How often instalment purchases are made (0 to 1) |
| CASH_ADVANCE_FREQUENCY | How often cash advance is taken (0 to 1) |
| CASH_ADVANCE_TRX | Number of cash advance transactions |
| PURCHASES_TRX | Number of purchase transactions |
| CREDIT_LIMIT | Credit limit of the customer |
| PAYMENTS | Total payments made by the customer |
| MINIMUM_PAYMENTS | Minimum payments due |
| PRC_FULL_PAYMENT | Percentage of full payment paid (0 to 1) |
| TENURE | Number of months as a card holder |

## 3. Import Libraries

We import the tools we need:
- **pandas, numpy** for data handling
- **matplotlib, seaborn** for charts
- **scikit-learn** for scaling, the three clustering algorithms, PCA and evaluation metrics
- **scipy** for drawing the dendrogram

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 4. Load and Understand the Data

First we load the CSV file and look at it. Always start by asking: how big is the data, what do the columns look like, and what are the data types?

In [ ]:

import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "CC GENERAL.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "arjunbhasin2013/ccdata",
  file_path,

)

df.head()

/tmp/ipykernel_501/219708100.py:8: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'ccdata' dataset.


,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


## 5. Data Cleaning

### 5.1 Check missing values

In [ ]:
df.isnull().sum()

,0
CUST_ID,0
BALANCE,0
BALANCE_FREQUENCY,0
PURCHASES,0
ONEOFF_PURCHASES,0
INSTALLMENTS_PURCHASES,0
CASH_ADVANCE,0
PURCHASES_FREQUENCY,0
ONEOFF_PURCHASES_FREQUENCY,0
PURCHASES_INSTALLMENTS_FREQUENCY,0


### 5.2 Check duplicates and drop the ID column

`CUST_ID` is only a label. It carries no behaviour information, so we remove it before clustering. If we kept it, the algorithm would try to use it.

In [ ]:
df['CREDIT_LIMIT'].fillna(df['CREDIT_LIMIT'].median(), inplace = True)
df['MINIMUM_PAYMENTS'].fillna(df['MINIMUM_PAYMENTS'].median(), inplace = True)

df.isnull().sum()

df.duplicated().sum()
df = df.drop('CUST_ID', axis = 1)

/tmp/ipykernel_501/2405840231.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['CREDIT_LIMIT'].fillna(df['CREDIT_LIMIT'].median(), inplace = True)
/tmp/ipykernel_501/2405840231.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, 

## 6. Exploratory Data Analysis (EDA)

### 6.1 Distribution of key features

## 7. Prepare the Data

All three algorithms measure the **distance** between customers. So we do two things first:

1. **Log transform** (`np.log1p`) shrinks the very large values so they no longer dominate.
2. **Scaling** (`StandardScaler`) puts every column on the same scale. Without it, `BALANCE` (in thousands) would count far more than `PURCHASES_FREQUENCY` (between 0 and 1).

## 8. Algorithm 1: K-Means
Inertia measures how close data points are to their assigned cluster centre (Centroid).

Simple Hinglish mein:

Inertia batati hai ki ek cluster ke andar customers apne cluster ke centre se kitne door ya paas hain.
- Low Inertia = Customers apne cluster centre ke paas hain.
- High Inertia = Customers apne cluster centre se door hain.

## Step 2: Add cluster labels to DataFrame
## Step 3: Check clusters
## Step 4: View data with clusters